The first cell is the api article fetcher
    - make sure you have your .env api key
    - change your START_DATE and END_DATE (I added the dates in comment to make it easier for you)
    - I(Victor) used PAGES = 70 and LIMIT = 5 (free plan limited to 3), to get 70*3=210 articles
    - That's it.

The second cell is to convert json to csv
    - Just run the cell (make sure that the above api article fetcher cell was run, in order for cached variables to exist)

Output files are saved in `data/raw/json_data/` and `data/raw/csv_data/`.

In [2]:
import http.client, urllib.parse,json
import os
from dotenv import load_dotenv
import time
from pathlib import Path


'''
THIS BLOCK WILL FETCH OUR JSON DATA WITH THENEWSAPI.
'''

conn = http.client.HTTPSConnection('api.thenewsapi.com')

#CHANGE THESE VARS TO YOUR OWN 
load_dotenv()
API_KEY = os.getenv("API_KEY") # you need a .env file with API_KEY=[your key] in its content

#Y: 2025-01-01 and 2025-06-24
#H: 2025-06-25 and 2025-12-17
#T: 2025-12-18 and 2026-06-01
#V: 2026-06-02 and 2026-11-15
START_DATE = os.getenv("START_DATE") #yyyy-mm-dd
END_DATE = os.getenv("END_DATE") #yyyy-mm-dd

PAGES = 70 #number of times [limit] amount of articles is fetched (70 means 210 articles for limit=3)
LIMIT = 5 #max is 3 for our free plan, so you don't have to change this.
SEARCH = 'Mamdani'

articles = []
for page in range(1, PAGES + 1):
    params = urllib.parse.urlencode({
        'api_token': API_KEY,
        'search': SEARCH,
        'locale': 'us,ca,mx',
        'language': 'en',
        'published_after': START_DATE,
        'published_before': END_DATE,
        'limit': LIMIT,
        'page': page
    
        })

    conn.request('GET', '/v1/news/all?{}'.format(params))

    res = conn.getresponse()

    if res.status != 200:
        print("API error:",  res.status, res.reason)
        print(res.read().decode('utf-8'))
        break

    data = res.read()
    results = json.loads(data.decode('utf-8'))
    # print("API metadata:", results.get("meta", {}))
    page_articles = results.get('data', [])
    articles.extend(page_articles)

    if len(articles) % 20 == 0:
        print(f"PAGE {page}: {len(articles)} articles have been collected")

    if len(articles) % 60 == 0:
        print(f"{len(articles)} collected, waiting 1 minute for api request timer reset...")
        time.sleep(60)

conn.close()

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
JSON_DIR = DATA_DIR / "raw" / "json_data"
CSV_DIR = DATA_DIR / "raw" / "csv_data"

# Create directories if they don't exist
for folder in (JSON_DIR, CSV_DIR, DATA_DIR / "cleaned", DATA_DIR / "annotated"):
    folder.mkdir(parents=True, exist_ok=True)

JSON_OUT = JSON_DIR / f"{START_DATE}__{END_DATE}.json"

with open(JSON_OUT, 'w', encoding='utf-8') as fo:
    json.dump(articles, fo, indent=4, ensure_ascii=False)

PAGE 20: 60 articles have been collected
60 collected, waiting 1 minute for api request timer reset...
PAGE 40: 120 articles have been collected
120 collected, waiting 1 minute for api request timer reset...
PAGE 60: 180 articles have been collected
180 collected, waiting 1 minute for api request timer reset...


In [3]:
import pandas as pd
'''
CONVERT THE JSON DATA TO CSV FORMAT
'''
#BELOW IS NOT NEEDED IF YOU JUST RAN THE JSON FETCHER FOR ARTICLE
# with open(JSON_OUT, 'r', encoding='utf-8') as f:
#     articles_data = json.load(f)

df = pd.DataFrame(articles)

CSV_OUT = CSV_DIR / f"{START_DATE}__{END_DATE}.csv"
df.to_csv(CSV_OUT, index=False, encoding='utf-8-sig')

print(f"Successfully saved {len(df)} articles to {CSV_OUT}")

Successfully saved 210 articles to c:\Users\grace\OneDrive\Bureau\McGill\COMP 370\COMP-370-Final-Project\data\raw\csv_data\2025-01-01__2025-06-24.csv
